# Función de costo en regresión lineal

Imagina que quieres predecir la calificación de un examen a partir de las horas
que alguien estudió. Vas a proponer una recta que convierta "horas" en
"calificación estimada". La pregunta de este notebook es: **¿cómo le pones un
número a qué tan buena o mala es esa recta?** Ese número es la *función de
costo*, y es lo que un algoritmo de entrenamiento intenta hacer lo más pequeño
posible.

> Requisito: si todavía no tienes claro qué son los parámetros `w` y `b` de una
> recta, revisa primero
> [`02_parametros_w_y_b.ipynb`](02_parametros_w_y_b.ipynb); aquí asumimos que ya
> entiendes que $\hat y = wx + b$ es "pendiente por x, más un desplazamiento".
> Aquí programamos el mecanismo de entrenamiento a mano; si primero quieres ver
> cómo se hace esto mismo con una librería en la práctica, revisa
> [`00_introduccion_y_scikit_learn.ipynb`](00_introduccion_y_scikit_learn.ipynb).

In [1]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from scipy.special import huber
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

datos = pl.DataFrame({
    "horas_estudio": [0, 1, 2, 3, 4, 5, 6],
    "calificacion": [5, 12, 22, 26, 38, 43, 48],
})

def predecir(x, w, b):
    return w * np.asarray(x) + b

# Una recta razonable, elegida a ojo, todavía no es la "mejor" recta.
w_ejemplo, b_ejemplo = 7.0, 6.0
datos = datos.with_columns(
    pl.Series("prediccion", predecir(datos["horas_estudio"], w_ejemplo, b_ejemplo))
).with_columns(
    (pl.col("calificacion") - pl.col("prediccion")).alias("residuo")
)
datos

horas_estudio,calificacion,prediccion,residuo
i64,i64,f64,f64
0,5,6.0,-1.0
1,12,13.0,-1.0
2,22,20.0,2.0
3,26,27.0,-1.0
4,38,34.0,4.0
5,43,41.0,2.0
6,48,48.0,0.0


## 1. Antes de la fórmula: ¿qué es un error aquí?

Para cada estudiante, la recta predice una calificación (`prediccion`) que casi
nunca coincide exactamente con la real (`calificacion`). La diferencia entre
ambas se llama **residuo**:

$$r_i = y_i - \hat y_i$$

- $y_i$ es el valor real de la observación $i$ (la calificación verdadera).
- $\hat y_i$ (se lee "y con techo" o "y gorro") es lo que predijo el modelo.
- $r_i$ es simplemente "cuánto te faltó o te sobró" en esa predicción.

Antes de calcular nada, mira el residuo directamente en la gráfica: es el largo
del segmento vertical entre cada punto real y la recta.

In [2]:
x = datos["horas_estudio"].to_numpy()
y = datos["calificacion"].to_numpy()
y_hat = datos["prediccion"].to_numpy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y_hat, mode="lines", name="predicción (recta)", line={"color": "#1f77b4"}))
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="calificación real", marker={"size": 11, "color": "black"}))
for xi, yi, yhi in zip(x, y, y_hat):
    fig.add_trace(go.Scatter(
        x=[xi, xi], y=[yhi, yi], mode="lines",
        line={"color": "#d62728", "dash": "dot"}, showlegend=False,
    ))
fig.add_annotation(x=x[-2], y=(y[-2] + y_hat[-2]) / 2, text="residuo", showarrow=True, arrowhead=2, ax=40, ay=0)
fig.update_layout(
    title="El residuo es la distancia vertical entre el dato real y la recta",
    xaxis_title="Horas de estudio", yaxis_title="Calificación",
)
fig.show()

Fíjate en dos cosas antes de seguir:

- Algunos residuos son positivos (la recta predijo de menos) y otros negativos
  (predijo de más). Si simplemente los sumáramos, unos cancelarían a otros y
  podríamos obtener un total cercano a cero aunque la recta esté bastante mal.
- Necesitamos una forma de medir el **tamaño** del error sin que el signo lo
  esconda, y que además nos diga qué tan grave es cada error.

In [3]:
suma_residuos = datos["residuo"].sum()
suma_residuos_absolutos = datos["residuo"].abs().sum()
print(f"Suma de los residuos (con signo):    {suma_residuos:6.2f}  <- engañosamente pequeña")
print(f"Suma de los residuos absolutos:       {suma_residuos_absolutos:6.2f}  <- refleja mejor el error total")

Suma de los residuos (con signo):      5.00  <- engañosamente pequeña
Suma de los residuos absolutos:        11.00  <- refleja mejor el error total


## 2. Por qué elevamos el residuo al cuadrado

Hay más de una manera razonable de "quitarle el signo" a un residuo:

- Tomar su **valor absoluto** $|r|$: cada unidad de error pesa lo mismo, sin
  importar qué tan grande sea.
- Elevarlo **al cuadrado** $r^2$: los errores grandes pesan mucho más que los
  pequeños.

Grafiquemos ambas opciones como "penalización" en función del tamaño del
residuo, para decidir con los ojos antes que con la fórmula.

In [4]:
r = np.linspace(-6, 6, 200)
penalizaciones = pl.DataFrame({
    "residuo": np.concatenate([r, r]),
    "penalizacion": np.concatenate([np.abs(r), r ** 2]),
    "tipo": ["valor absoluto |r|"] * len(r) + ["cuadrado r²"] * len(r),
})
fig = px.line(
    penalizaciones, x="residuo", y="penalizacion", color="tipo",
    title="Elevar al cuadrado castiga mucho más fuerte los errores grandes",
)
fig.update_layout(xaxis_title="tamaño del residuo (r)", yaxis_title="penalización")
fig.show()

Con el cuadrado, un residuo de 4 pesa **16 veces** más que un residuo de 1 (no
4 veces); con el valor absoluto, pesa exactamente 4 veces más. Esa diferencia
importa: al cuadrado, el modelo se preocupa desproporcionadamente por evitar
errores grandes, lo cual suele ser deseable (un error enorme en una sola
predicción es peor que muchos errores pequeños repartidos). Además, la curva
$r^2$ es una sola "U" suave, sin ningún pico ni esquina — más adelante, cuando
hablemos de descenso de gradiente, esa suavidad es justo lo que hace posible
calcular una dirección de mejora en cualquier punto.

## 3. La fórmula: error cuadrático medio (MSE)

Con esta idea, el **MSE** (*Mean Squared Error*) promedia el cuadrado de todos
los residuos:

$$J(w,b) = \frac{1}{n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Leyendo la fórmula término a término:

- $n$ es el número de observaciones (aquí, 7 estudiantes).
- $\sum_{i=1}^n$ dice "suma esto para cada observación, de la 1 a la n".
- $(y_i - \hat y_i)^2$ es el residuo de esa observación, al cuadrado —
  justo lo que graficamos arriba.
- Dividir por $n$ convierte la suma en un **promedio**: así el MSE no crece
  solo porque tengas más datos, sino porque las predicciones son peores.

$J(w,b)$ se escribe como función de $w$ y $b$ porque, para los mismos datos,
distintas rectas (distintos $w,b$) producen distintos residuos y por tanto
distinto costo.

In [5]:
costo_mse = mean_squared_error(y, y_hat)
costo_rmse = root_mean_squared_error(y, y_hat)
print(f"MSE:  {costo_mse:.2f}  (unidades: 'calificación al cuadrado', difícil de interpretar)")
print(f"RMSE: {costo_rmse:.2f}  (unidades: 'calificación', misma escala que y)")

MSE:  3.86  (unidades: 'calificación al cuadrado', difícil de interpretar)
RMSE: 1.96  (unidades: 'calificación', misma escala que y)


El **RMSE** (*Root Mean Squared Error*) es simplemente la raíz cuadrada del
MSE. Se usa mucho para comunicar resultados porque, al deshacer el cuadrado,
vuelve a las unidades originales de $y$: si predices calificaciones, el RMSE se
lee en puntos de calificación; si predices minutos, se lee en minutos.

## 4. MAE y Huber: otras prioridades para el mismo problema

El **MAE** (*Mean Absolute Error*) usa la otra penalización que graficamos:

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

Como cada unidad de error pesa igual, el MAE es menos sensible a un dato
atípico (*outlier*) que el MSE. **Huber Loss** es un punto intermedio: se
comporta como el cuadrado cerca de cero y como el valor absoluto lejos de cero
(lo veremos a fondo en
[`03_huber_loss_en_profundidad.ipynb`](03_huber_loss_en_profundidad.ipynb)).
MSE y MAE se estudian cada una en su propio notebook de profundización —
[`04_mse_en_profundidad.ipynb`](04_mse_en_profundidad.ipynb) y
[`05_mae_en_profundidad.ipynb`](05_mae_en_profundidad.ipynb) — con ejemplos
simples y reales de cuándo conviene cada una.
Comparemos las tres penalizaciones juntas:

In [6]:
delta = 5.0
r = np.linspace(-12, 12, 300)
comparacion = pl.DataFrame({
    "residuo": np.concatenate([r, r, r]),
    "penalizacion": np.concatenate([r ** 2 / 2, np.abs(r), huber(delta, r)]),
    "perdida": ["MSE (r²/2)"] * len(r) + ["MAE (|r|)"] * len(r) + [f"Huber (δ={delta:.0f})"] * len(r),
})
px.line(
    comparacion, x="residuo", y="penalizacion", color="perdida",
    title="MSE crece más rápido que MAE; Huber sigue a MSE cerca de 0 y a MAE lejos",
).show()

### Caso real: estimar tiempos de entrega de comida

Una aplicación predice los minutos que tardará un pedido. Cinco pedidos
normales tienen errores pequeños, pero uno sufrió un accidente de tráfico y
tardó 150 minutos más de lo previsto. Ese evento es real, pero no representa
una entrega habitual. Calculemos MAE y MSE sobre el mismo modelo para ver cómo
reacciona cada uno ante ese único dato extremo.

In [7]:
entregas = pl.DataFrame({
    "pedido": ["P1", "P2", "P3", "P4", "P5", "P6: accidente"],
    "minutos_reales": [28, 35, 42, 50, 60, 205],
    "minutos_predichos": [30, 39, 44, 47, 56, 55],
}).with_columns(
    (pl.col("minutos_reales") - pl.col("minutos_predichos")).abs().alias("error_absoluto"),
    ((pl.col("minutos_reales") - pl.col("minutos_predichos")) ** 2).alias("error_cuadrado"),
)
entregas

pedido,minutos_reales,minutos_predichos,error_absoluto,error_cuadrado
str,i64,i64,i64,i64
"""P1""",28,30,2,4
"""P2""",35,39,4,16
"""P3""",42,44,2,4
"""P4""",50,47,3,9
"""P5""",60,56,4,16
"""P6: accidente""",205,55,150,22500


In [8]:
fig = px.bar(
    entregas.unpivot(index="pedido", on=["error_absoluto", "error_cuadrado"], variable_name="tipo", value_name="contribucion"),
    x="pedido", y="contribucion", color="tipo", barmode="group",
    title="El accidente domina el error cuadrado, pero pesa mucho menos en el error absoluto",
)
fig.show()

El pedido P6 aporta una sola barra enorme en `error_cuadrado`: ese único dato
puede dominar por completo el entrenamiento si usas MSE, arrastrando la recta
hacia intentar "acertar" ese accidente a costa de los demás pedidos. En
`error_absoluto` el mismo dato pesa mucho más que los otros, pero no los
aplasta del todo. Esta es la razón práctica para elegir MAE, RMSE o Huber según
el problema: no es solo una preferencia matemática, es una decisión sobre qué
tan grave permites que sea un error atípico.

## 5. El costo como una superficie: por qué se puede minimizar

Cada combinación de pendiente $w$ y desplazamiento $b$ produce una recta
distinta y, por tanto, un MSE distinto. Si fijamos $b$ y probamos muchos
valores de $w$, obtenemos una curva de costo con forma de valle:

In [9]:
b_fijo = 6.0
valores_w = np.linspace(-2, 14, 200)
costo_por_w = pl.DataFrame({
    "w": valores_w,
    "mse": [mean_squared_error(y, predecir(x, w_candidato, b_fijo)) for w_candidato in valores_w],
})
mejor_fila = costo_por_w.sort("mse").head(1)
fig = px.line(costo_por_w, x="w", y="mse", title=f"Costo al variar w, con b fijo en {b_fijo:.0f}: un único valle")
fig.add_vline(x=mejor_fila["w"][0], line_dash="dash", annotation_text="mínimo")
fig.show()

Esa forma de "un solo valle, sin baches" se llama **convexa**: imagina una
pelota rodando dentro de un tazón — sin importar desde dónde la sueltes, siempre
termina en el mismo fondo, porque no hay otros huecos donde pueda quedar
atrapada. El MSE de una regresión lineal siempre tiene esta forma (en más
dimensiones, un "tazón" multidimensional). Esa propiedad es la que hace posible
encontrar el mínimo de forma confiable con el método que sigue.

Si en lugar de fijar $b$ dejamos que $w$ y $b$ varíen juntos, el costo forma una
superficie. Podemos representarla como un mapa de calor, donde cada color es un
valor de MSE:

In [10]:
rejilla_w = np.linspace(0, 14, 70)
rejilla_b = np.linspace(-6, 18, 70)
superficie = pl.DataFrame([
    {"w": w_c, "b": b_c, "mse": mean_squared_error(y, predecir(x, w_c, b_c))}
    for w_c in rejilla_w
    for b_c in rejilla_b
])
fig_superficie = px.density_heatmap(
    superficie, x="w", y="b", z="mse", histfunc="avg",
    color_continuous_scale="Viridis_r", nbinsx=70, nbinsy=70,
    title="Mapa de costo MSE(w, b): la zona oscura es el fondo del tazón",
)
fig_superficie.show()

## 6. Encontrar el mínimo: descenso de gradiente

Ya vimos *que* existe un único mínimo. Ahora veamos *cómo* un algoritmo lo
encuentra sin tener que probar miles de combinaciones de $(w,b)$ a fuerza
bruta.

### 6.1 Qué es una "pendiente" en la curva de costo

Mira de nuevo la curva de costo al variar $w$. En cualquier punto de esa curva
puedes trazar una línea recta que la toca justo ahí y sigue su misma
inclinación local: esa línea es la **tangente**, y su inclinación es la
**derivada** (o pendiente) del costo en ese punto.

- Si la tangente **baja hacia la derecha** (pendiente negativa), moverte hacia
  la derecha (aumentar $w$) reduce el costo.
- Si la tangente **sube hacia la derecha** (pendiente positiva), moverte hacia
  la izquierda (disminuir $w$) reduce el costo.
- En el fondo del valle, la tangente es horizontal (pendiente cero): ahí ya no
  puedes mejorar moviéndote en ninguna dirección — es el mínimo.

In [11]:
def costo_de_w(w_candidato):
    return mean_squared_error(y, predecir(x, w_candidato, b_fijo))

def pendiente_de_w(w_candidato, h=1e-4):
    return (costo_de_w(w_candidato + h) - costo_de_w(w_candidato - h)) / (2 * h)

puntos_ejemplo = [1.0, 7.0, 12.0]
fig = px.line(costo_por_w, x="w", y="mse", title="La tangente indica hacia dónde moverse para bajar el costo")
for w0 in puntos_ejemplo:
    pendiente = pendiente_de_w(w0)
    costo0 = costo_de_w(w0)
    ancho = 2.0
    fig.add_trace(go.Scatter(
        x=[w0 - ancho, w0 + ancho],
        y=[costo0 - pendiente * ancho, costo0 + pendiente * ancho],
        mode="lines", line={"color": "#d62728", "dash": "dot"}, showlegend=False,
    ))
    fig.add_trace(go.Scatter(x=[w0], y=[costo0], mode="markers", marker={"size": 10, "color": "#d62728"}, showlegend=False))
fig.show()

Esa es exactamente la idea del **gradiente**: la derivada del costo respecto a
cada parámetro, evaluada en el punto actual. Para todos los datos a la vez, las
fórmulas son:

$$\frac{\partial J}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n}x_i(y_i-\hat{y}_i), \qquad \frac{\partial J}{\partial b} = -\frac{2}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)$$

- $\frac{\partial J}{\partial w}$ se lee "derivada parcial de J respecto a w":
  cuánto y hacia dónde cambia el costo si movemos solo $w$ un poquito.
- El signo negativo dentro de la suma viene de derivar $(y_i - \hat y_i)^2$: al
  final, la fórmula compara la dirección del error con la entrada $x_i$ (o con
  1, para $b$).

El **descenso de gradiente** actualiza los parámetros dando un paso en la
dirección contraria al gradiente (porque el gradiente apunta hacia donde el
costo *sube*, y queremos bajar):

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w}, \qquad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

$\alpha$ (la **tasa de aprendizaje**) controla qué tan grande es cada paso.

In [12]:
w, b = 0.0, 0.0
tasa_aprendizaje = 0.02
historial = []

for epoca in range(120):
    pred = w * x + b
    error = y - pred
    grad_w = -2 * np.mean(x * error)
    grad_b = -2 * np.mean(error)
    w -= tasa_aprendizaje * grad_w
    b -= tasa_aprendizaje * grad_b
    historial.append({"epoca": epoca, "w": w, "b": b, "mse": mean_squared_error(y, w * x + b)})

historial = pl.DataFrame(historial)
print(f"Parámetros finales: w={w:.2f}, b={b:.2f}")
historial.head(5)

Parámetros finales: w=7.60, b=4.66


epoca,w,b,mse
i64,f64,f64,f64
0,4.508571,1.108571,207.092408
1,6.539657,1.631771,47.418779
2,7.451794,1.890313,14.774601
3,7.858595,2.029057,8.046992
4,8.03721,2.113434,6.608133


### 6.2 Ver la trayectoria completa, no solo el resultado final

En vez de mirar únicamente el valor final de $(w,b)$, dibujemos **cada paso**
del descenso de gradiente sobre el mismo mapa de calor de antes. Así se ve
literalmente cómo el algoritmo "camina cuesta abajo" hacia el fondo del
tazón.

In [13]:
pasos_a_mostrar = historial.gather_every(4)
fig = px.density_heatmap(
    superficie, x="w", y="b", z="mse", histfunc="avg",
    color_continuous_scale="Viridis_r", nbinsx=70, nbinsy=70,
    title="Trayectoria del descenso de gradiente sobre el mapa de costo",
)
fig.add_trace(go.Scatter(
    x=pasos_a_mostrar["w"], y=pasos_a_mostrar["b"], mode="lines+markers",
    marker={"size": 6, "color": "white"}, line={"color": "white"}, name="trayectoria",
))
fig.add_trace(go.Scatter(x=[w], y=[b], mode="markers", marker={"size": 13, "color": "red", "symbol": "star"}, name="parámetros finales"))
fig.show()

In [14]:
px.line(
    historial, x="epoca", y="mse",
    title="El costo baja rápido al inicio y luego se aplana cerca del mínimo",
).show()

Los primeros pasos reducen el costo muy rápido, porque la pendiente es
pronunciada lejos del mínimo. Cerca del fondo, la pendiente se acerca a cero,
los pasos se vuelven más pequeños y el costo casi no cambia: el algoritmo está
convergiendo. En la práctica, `scikit-learn` resuelve esta optimización
internamente (o de forma analítica, para regresión lineal simple); programar
el bucle manualmente aquí es solo para ver el mecanismo por dentro. La
pregunta de exactamente cuándo existe esa solución analítica y cuándo hace
falta descenso de gradiente se responde a fondo en
[`06_minimos_cuadrados_vs_descenso_de_gradiente.ipynb`](06_minimos_cuadrados_vs_descenso_de_gradiente.ipynb).

## 7. Ideas clave

- El residuo mide cuánto se equivocó una predicción; el MSE promedia esos
  residuos elevados al cuadrado, lo que penaliza fuerte los errores grandes.
- El RMSE es MSE en las unidades originales de $y$, más fácil de comunicar.
- MAE y Huber son alternativas menos sensibles a valores atípicos; la elección
  depende de qué tan grave consideras un error extremo.
- El costo de una regresión lineal, visto como función de $(w,b)$, tiene forma
  convexa: un único valle, sin mínimos falsos donde el algoritmo pueda
  quedarse atrapado.
- El descenso de gradiente usa la pendiente local del costo para decidir hacia
  dónde moverse, dando pasos hasta acercarse al fondo del valle.

**Ejercicio:** cambia `w_ejemplo` y `b_ejemplo` en la primera celda de código
por valores muy alejados de la recta real (por ejemplo `w=1, b=0`) y vuelve a
ejecutar el notebook. Antes de mirar los resultados, predice: ¿el residuo iba a
ser más grande o más pequeño? ¿La trayectoria del descenso de gradiente
necesitará más o menos pasos para llegar al mínimo?